In [2]:
!pip install transformers datasets seqeval evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=1d2cffed81564d30d221c0f5e8d78194f731df79c93e1a5418a6cd0179c5c149
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

In [6]:
import os
import logging
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"
from transformers.utils import logging
logging.set_verbosity_error()
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

In [7]:
%pip install -q transformers datasets seqeval evaluate accelerate --quiet
%pip install -q nltk --quiet

In [8]:
import numpy as np
import torch
import nltk
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline,
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {device}")
print(f"✅ PyTorch: {torch.__version__}")

✅ Device: cpu
✅ PyTorch: 2.10.0+cpu


In [9]:
data = [
    {
        "tokens": ["John", "works", "at", "Google"],
        "pos_tags": ["NNP", "VBZ", "IN", "NNP"],
        "chunk_tags": ["B-NP", "B-VP", "B-PP", "B-NP"],
    },
    {
        "tokens": ["She", "loves", "machine", "learning"],
        "pos_tags": ["PRP", "VBZ", "NN", "NN"],
        "chunk_tags": ["B-NP", "B-VP", "I-NP", "I-NP"],
    },
    {
        "tokens": ["I", "am", "learning", "NLP"],
        "pos_tags": ["PRP", "VBP", "VBG", "NNP"],
        "chunk_tags": ["B-NP", "B-VP", "I-VP", "B-NP"],
    },
]

In [10]:
dataset = Dataset.from_list(data)

dataset = DatasetDict({
    "train": dataset,
    "validation": dataset,
    "test": dataset
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 3
    })
    validation: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 3
    })
    test: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 3
    })
})


In [11]:
pos_labels = list(set(label for example in data for label in example["pos_tags"]))
chunk_labels = list(set(label for example in data for label in example["chunk_tags"]))

print("POS Labels:", pos_labels)
print("Chunk Labels:", chunk_labels)

POS Labels: ['VBP', 'PRP', 'NN', 'VBG', 'VBZ', 'NNP', 'IN']
Chunk Labels: ['B-VP', 'I-NP', 'B-NP', 'I-VP', 'B-PP']


In [12]:
pos_label2id = {label: i for i, label in enumerate(pos_labels)}
pos_id2label = {i: label for label, i in pos_label2id.items()}

chunk_label2id = {label: i for i, label in enumerate(chunk_labels)}
chunk_id2label = {i: label for label, i in chunk_label2id.items()}

print("POS mapping:", pos_label2id)
print("Chunk mapping:", chunk_label2id)

POS mapping: {'VBP': 0, 'PRP': 1, 'NN': 2, 'VBG': 3, 'VBZ': 4, 'NNP': 5, 'IN': 6}
Chunk mapping: {'B-VP': 0, 'I-NP': 1, 'B-NP': 2, 'I-VP': 3, 'B-PP': 4}


In [13]:
def encode_labels(example):
    example["pos_tags"] = [pos_label2id[label] for label in example["pos_tags"]]
    example["chunk_tags"] = [chunk_label2id[label] for label in example["chunk_tags"]]
    return example

dataset = dataset.map(encode_labels)

dataset["train"][0]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

{'tokens': ['John', 'works', 'at', 'Google'],
 'pos_tags': [5, 4, 6, 5],
 'chunk_tags': [2, 0, 4, 2]}

In [14]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [15]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()

    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(example["pos_tags"][word_idx])  # POS tagging
        else:
            labels.append(-100)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [16]:
tokenized_datasets = dataset.map(tokenize_and_align_labels)

tokenized_datasets["train"][0]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

{'tokens': ['John', 'works', 'at', 'Google'],
 'pos_tags': [5, 4, 6, 5],
 'chunk_tags': [2, 0, 4, 2],
 'input_ids': [101, 2198, 2573, 2012, 8224, 102],
 'token_type_ids': [0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1],
 'labels': [-100, 5, 4, 6, 5, -100]}

In [17]:
num_labels = len(pos_labels)

print("Number of labels:", num_labels)

Number of labels: 7


In [18]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=pos_id2label,
    label2id=pos_label2id
)

model.to(device)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [19]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [20]:
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=3,   # small dataset → slightly more epochs
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    learning_rate=2e-5,

    logging_steps=10,

    do_train=True,
    do_eval=True
)

In [21]:
metric = evaluate.load("seqeval")

In [22]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        curr_preds = []
        curr_labels = []

        for p_val, l_val in zip(pred, lab):
            if l_val != -100:
                curr_preds.append(pos_id2label[p_val])
                curr_labels.append(pos_id2label[l_val])

        true_predictions.append(curr_preds)
        true_labels.append(curr_labels)

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [23]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [24]:
trainer.train()


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '18.62', 'train_samples_per_second': '0.483', 'train_steps_per_second': '0.322', 'train_loss': '1.773', 'epoch': '3'}


TrainOutput(global_step=6, training_loss=1.772679328918457, metrics={'train_runtime': 18.6184, 'train_samples_per_second': 0.483, 'train_steps_per_second': 0.322, 'train_loss': 1.772679328918457, 'epoch': 3.0})

In [25]:
results = trainer.evaluate()

print("Evaluation Results:")
for key, value in results.items():
    print(f"{key}: {value}")

{'eval_loss': '1.494', 'eval_precision': '0.8182', 'eval_recall': '0.8182', 'eval_f1': '0.8182', 'eval_accuracy': '0.75', 'eval_runtime': '0.2077', 'eval_samples_per_second': '14.44', 'eval_steps_per_second': '9.629', 'epoch': '3'}
Evaluation Results:
eval_loss: 1.4939326047897339
eval_precision: 0.8181818181818182
eval_recall: 0.8181818181818182
eval_f1: 0.8181818181818182
eval_accuracy: 0.75
eval_runtime: 0.2077
eval_samples_per_second: 14.443
eval_steps_per_second: 9.629
epoch: 3.0


In [26]:
nlp = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if device == "cuda" else -1
)

In [27]:
sentence = "John works at Google in California"

output = nlp(sentence)

for token in output:
    print(token)

{'entity': 'IN', 'score': np.float32(0.2091287), 'index': 1, 'word': 'john', 'start': 0, 'end': 4}
{'entity': 'VBZ', 'score': np.float32(0.30204567), 'index': 2, 'word': 'works', 'start': 5, 'end': 10}
{'entity': 'IN', 'score': np.float32(0.20085183), 'index': 3, 'word': 'at', 'start': 11, 'end': 13}
{'entity': 'NNP', 'score': np.float32(0.18978263), 'index': 4, 'word': 'google', 'start': 14, 'end': 20}
{'entity': 'VBZ', 'score': np.float32(0.22606672), 'index': 5, 'word': 'in', 'start': 21, 'end': 23}
{'entity': 'NNP', 'score': np.float32(0.21562703), 'index': 6, 'word': 'california', 'start': 24, 'end': 34}


In [28]:
for token in output:
    print(f"{token['word']} → {token['entity']}")

john → IN
works → VBZ
at → IN
google → NNP
in → VBZ
california → NNP


POS Tagging vs Chunking — Comparison

Part-of-Speech (POS) Tagging:
Assigns grammatical labels to each word
Examples:
Noun (NN)
Verb (VB)
Adjective (JJ)
Focus: Word-level classification
Complexity: Easier
Chunking (Phrase Detection):
Groups words into meaningful phrases

Examples:

Noun Phrase (NP)

Verb Phrase (VP)

Uses BIO tagging:

B → Beginning

I → Inside

O → Outside

Focus: Phrase-level understanding

Complexity: Higher than POS tagging

🧪 Observations

POS tagging is simpler because it labels individual words

Chunking is more complex because it depends on context and grouping

Transformer models like BERT perform well for both tasks due to contextual understanding

Token classification requires careful handling of subwords and label alignment

⚠️ Challenges Faced

Handling tokenization where one word splits into multiple subwords

Aligning labels correctly using -100 for ignored tokens

Managing compatibility issues with datasets library

Training on CPU required optimization and smaller dataset usage

✅ Conclusion

Successfully implemented token classification using BERT
Performed POS tagging using transformer-based model

Understood:

Tokenizationand label alignment

Transformer fine-tuning for sequence
labeling

Evaluation using sequence metrics (seqeval)

BERT demonstrates strong capability in understanding context at token level

🚀 Future Improvements

Train on larger datasets like CoNLL-2003

Use GPU for faster training

Extend to chunking model training separately

Try advanced models like DistilBERT or RoBERTa